In [ ]:
# Transient-dynamics supplementary figure (parallel to 09_Supp_ring_attractor.ipynb)
import os
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from fig_utils.basii import compute_orthogonal_subspace
from vi_rnn.data_utils import make_all_trials, stim_end_bins
from fig_utils.perturbation import (
    compute_perturbation_distance_stats,
    generate_w_perturb_x,
    perturbation_goal_bounds,
    position_latent_indices,
)
from fig_utils.plots import (
    plot_basis_3d_trajectories,
    plot_connectivity_matrix,
    plot_distance_moved_vs_class_mean,
    plot_perturbation_latent_snapshots_attractor,
    plot_spike_histogram_stats,
    plot_student_teacher_overview,
    plot_transient_xmode_readout,
    plot_unit_activity_by_stimulus,
    plot_unit_trials_overlay,
)
from fig_utils.spike_stats import spike_histogram_stats_from_array
from fig_utils.toy_models import (
    build_orthogonal_lrrnn,
    generate_student_teacher_dataset,
    init_transient_lrrnn,
)

cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=6))
%matplotlib inline

In [ ]:
save = False  # set True to write .npy files


# --- Transient toy low-rank RNN (methods-consistent) ---
P = 6
N = 500
rank = 20
n_nn_modes = rank // 2
dt = 0.05
tau = 0.3
g_ff = 2.0
g_out = 0.3
relu_bias = 0.0
sigma_dyn = 0.005
noise_scale = (
    1.0  # 1.0  # 0: no process noise and deterministic z0 ICs (ic_scale ignored)
)
sigma_I = 1.6
alpha_I = 0.9
d_scale = 0.1
T = int(5 / dt)
time = np.arange(T) * dt
seed = 0
rnn = init_transient_lrrnn(
    seed=seed,
    N=N,
    rank=rank,
    dt=dt,
    tau=tau,
    g_ff=g_ff,
    g_out=g_out,
    relu_bias=relu_bias,
    sigma_dyn=sigma_dyn,
    sigma_I=sigma_I,
    alpha_I=alpha_I,
    d_scale=d_scale,
)
M = rnn.M
N_vecs = rnn.N_vecs
I = rnn.I
J = rnn.J
M_inv = np.linalg.pinv(M)

In [ ]:
alpha = 1 - dt / tau
print(alpha)

In [ ]:
# --- Mode readout (kappa): x-mode cascade and y-mode cascade ---
input_start = int(0.5 / dt)
n_time_input = int(0.5 / dt)
thetas = np.linspace(0, 2 * np.pi, P, endpoint=False)
fig, ax = plt.subplots(figsize=(2, 1.3), dpi=300)
mode_colors_x = plt.cm.gray(np.linspace(0.2, 0.8, n_nn_modes - 1))
for p, theta in enumerate(thetas):
    u_demo = np.zeros((1, 2, T))
    u_demo[0, 0, input_start : input_start + n_time_input] = np.cos(theta)
    u_demo[0, 1, input_start : input_start + n_time_input] = np.sin(theta)
    X = rnn.simulate_x(u_demo, ic_scale=0.1)
    kappa = X[0].T @ M_inv.T
    for i in range(n_nn_modes - 1):
        ax.plot(
            time,
            kappa[:, i],
            color=mode_colors_x[i],
            lw=1.5,
            alpha=0.35 + 0.65 * (p == 0),
        )
ax.set_xlabel("time")
ax.set_ylabel("mode coeff")
ax.set_title("x-mode sequence")
plt.tight_layout()
fig, ax = plt.subplots(figsize=(2, 1.3), dpi=300)
mode_colors_y = plt.cm.gray(np.linspace(0.2, 0.8, n_nn_modes))
for p, theta in enumerate(thetas):
    u_demo = np.zeros((1, 2, T))
    u_demo[0, 0, input_start : input_start + n_time_input] = np.cos(theta)
    u_demo[0, 1, input_start : input_start + n_time_input] = np.sin(theta)
    X = rnn.simulate_x(u_demo, ic_scale=0.1)
    kappa = X[0].T @ M_inv.T
    for i in range(n_nn_modes, 2 * n_nn_modes):
        ax.plot(
            time,
            kappa[:, i],
            color=mode_colors_y[i - n_nn_modes],
            lw=1.5,
            alpha=0.35 + 0.65 * (p == 0),
        )
ax.set_xlabel("time")
ax.set_ylabel("mode coeff")
ax.set_title("y-mode sequence")
plt.tight_layout()

In [ ]:
# --- Example plot for Supp fig
theta = 0
input_start = int(0.5 / dt)
n_time_input = int(0.5 / dt)
u_demo = np.zeros((1, 2, T))
u_demo[0, 0, input_start : input_start + n_time_input] = np.cos(theta)
u_demo[0, 1, input_start : input_start + n_time_input] = np.sin(theta)
X = rnn.simulate_x(u_demo, ic_scale=0.1)
kappa = X[0].T @ M_inv.T
plot_transient_xmode_readout(
    time,
    kappa[:, : n_nn_modes - 1],
    input_start_bin=input_start,
    dt=dt,
    box_w=1,
    box_h=0.5,
)

In [ ]:
# --- Example plot for fig
theta = 0
input_start = int(0.5 / dt)
n_time_input = int(0.5 / dt)
u_demo = np.zeros((1, 2, T))
u_demo[0, 0, input_start : input_start + n_time_input] = np.cos(theta)
u_demo[0, 1, input_start : input_start + n_time_input] = np.sin(theta)
X = rnn.simulate_x(u_demo, ic_scale=0.1)
kappa = X[0].T @ M_inv.T
plot_transient_xmode_readout(
    time,
    kappa[:, : n_nn_modes - 1],
    input_start_bin=input_start,
    dt=dt,
    box_w=1.4,
    box_h=0.9,
)

In [ ]:
plot_connectivity_matrix(J, N)

In [ ]:
trial_duration_s = 5.0
stim_onset_s = 0.55
stim_duration_s = 0.25
n_repeats = 50
task_params = {
    "incl_ns_stim": False,
    "incl_probe": False,
    "incl_cue": False,
    "incl_ramp": False,
}
u_master, _, _, _ = make_all_trials(
    task_params,
    dur=trial_duration_s,
    n_stim=P,
    n_pos=1,
    onset=stim_onset_s,
    stim_dur=stim_duration_s,
    bin_size=dt,
    interval_dur="mean",
    delay_dur="mean",
)
u = np.repeat(u_master, n_repeats, axis=0)
labels_p = np.repeat(np.arange(P), n_repeats)
n_steps = u.shape[-1]
Z, X = rnn.simulate(u, noise_scale=noise_scale, return_x=True)
Z_all = Z.reshape(P, n_repeats, rnn.dim_z, n_steps)
ys_all = X.reshape(P, n_repeats, N, n_steps)

In [ ]:
unit_ids = [
    np.random.choice(
        np.arange(pop_id * N // 4, (pop_id + 1) * N // 4), size=1, replace=False
    )[0]
    for pop_id in range(4)
]
plot_unit_trials_overlay(
    ys_all, unit_ids, stim_index=0, n_steps=n_steps, panel_gap_y=0.3
)

In [ ]:
# basii on condition-averaged latents z (dim_z, P, T) -> W_full is (dim_z, dim_z)
z_dpca = Z_all.mean(axis=1).transpose(1, 0, 2)  # (dim_z, P, n_steps)
time_windows = [[20, 60], [20, 60]]
n_pcs_time_basii = 1
(
    transform,
    inv_transform,
    W_enc,
    W_full,
    global_mean,
    scaling,
) = compute_orthogonal_subspace(
    z_dpca,
    marg_order=["t", "s1"],
    n_components=[n_pcs_time_basii, 2],
    time_windows=time_windows,
    center=True,
    center_marginals=True,
    soft_norm_constant=5.0,
    orthogonalize=True,
    standardize=False,
    basis="pca",
    lrr_reg=1e-5,
)
print("W_enc", W_enc.shape, "W_full", W_full.shape)

Z_basii = transform(z_dpca)  # (n_basii, P, n_steps)
Z_T = Z_basii.transpose(1, 0, 2)  # (P, n_basii, T) for 3D plot
traj_labels = np.arange(P)[:, None]
run_pyvista = True
if run_pyvista:
    plot_basis_3d_trajectories(
        Z_T,  # match notebook 9
        traj_labels,
        pos_ind=0,
        cmap=cmap,
        n_pcs_time=n_pcs_time_basii,
        bin_size=dt,
        plt_start=0,
        plt_end=n_steps - 40,
        jupyter_backend="static",
        window_size=(2000, 1400),
        show=True,
        xscale=1,
        yscale=1.5,
        zscale=1,
        tscale=1,
        stim_mark_times=[stim_onset_s],
        tube_radius=0.02,
        azimuth=5,
        elevation=-30,
        zoom=1.4,
        x_axis_scale=0.7,
        y_axis_scale=1,
        z_axis_scale=0.7,
        axis_line_width=8,
        stim_line_width=4,
        shadow_opacity=0.2,
        shadow_line_width=10,
    )

In [ ]:
rnn_orth = build_orthogonal_lrrnn(rnn, W_full, z_mean=global_mean, z_scale=scaling)
print(f"rnn_orth dim_z={rnn_orth.dim_z}")

# Same ``u`` and noise_scale as the hand-built demo above
Z_orth, _ = rnn_orth.simulate(u, noise_scale=noise_scale, return_x=False)
Z_T_orth = Z_orth.reshape(P, n_repeats, rnn_orth.dim_z, n_steps).mean(axis=1)

if run_pyvista:
    plot_basis_3d_trajectories(
        Z_T_orth,
        traj_labels,
        pos_ind=0,
        cmap=cmap,
        n_pcs_time=n_pcs_time_basii,
        bin_size=dt,
        plt_start=0,
        plt_end=n_steps - 30,
        jupyter_backend="static",
        window_size=(2000, 1400),
        show=True,
    )

In [ ]:
trial_duration_s = 5.0
n_repeats = 100
BINS_AFTER_LAST_STIM = 9
N_BINS_FOR_R = 15

u_master, _, _, _ = make_all_trials(
    task_params,
    dur=trial_duration_s,
    n_stim=P,
    n_pos=1,
    cue_dur=-1,
    bin_size=dt,
    interval_dur="mean",
    delay_dur="mean",
)
u = np.repeat(u_master, n_repeats, axis=0)
labels_p = np.repeat(np.arange(P), n_repeats)
n_steps = u.shape[-1]
last_stim_end = int(stim_end_bins(u).max())
t_perturb = last_stim_end + BINS_AFTER_LAST_STIM
t_move_start = t_perturb
t_move_end = t_move_start + N_BINS_FOR_R
rnn_orth.z0 = np.zeros(rnn_orth.dim_z)
Z_base = generate_w_perturb_x(rnn_orth, u=u, noise_scale=noise_scale)
Z_all = Z_base.reshape(P, n_repeats, rnn_orth.dim_z, n_steps)
# Stimulus plane in orth basis: indices n_pcs_time_basii + 2*pos (+0, +1)
pos = 0
z1, z2 = position_latent_indices(n_pcs_time_basii, pos)
perturb_inds = [z1, z2]
pert_z1 = perturbation_goal_bounds(Z_base[:, z1, t_perturb])
pert_z2 = perturbation_goal_bounds(Z_base[:, z2, t_perturb])
goals = np.random.rand(u.shape[0], 2)
goals[:, 0] = goals[:, 0] * (pert_z1[1] - pert_z1[0]) + pert_z1[0]
goals[:, 1] = goals[:, 1] * (pert_z2[1] - pert_z2[0]) + pert_z2[0]
pert_dir = rnn_orth.M
goal_amp = 1.0
Z_pert = generate_w_perturb_x(
    rnn_orth,
    u=u,
    noise_scale=noise_scale,
    perturb_at_t=t_perturb,
    perturb_weights=pert_dir,
    perturb_inds=perturb_inds,
    goal=goals,
    goal_amp=goal_amp,
    optogen=0.0,
)
Z_all_perturb = Z_pert.reshape(P, n_repeats, rnn_orth.dim_z, n_steps)

In [ ]:
P, n_repeats, dim_z, n_steps = Z_all.shape
Z = Z_all.reshape(-1, dim_z, n_steps)
Z_pert = Z_all_perturb.reshape(-1, dim_z, n_steps)
labels_pos = np.repeat(np.arange(P), n_repeats)
plot_ts = [t_perturb - 2, t_perturb, t_perturb + 10, t_move_end]
plot_perturbation_latent_snapshots_attractor(
    Z,
    Z_pert,
    labels_pos,
    z1=z1,
    z2=z2,
    cmap=cmap,
    plot_ts=plot_ts,
    highlight_cond=2,
    bin_size=dt,
    show=True,
)

In [ ]:
plot_ts

In [ ]:
t1 = t_move_start
t2 = t_move_end
# z1, z2, pos, n_pcs_time_basii from perturb cell
stats = compute_perturbation_distance_stats(
    Z,
    Z_pert,
    labels_pos,
    z1=z1,
    z2=z2,
    t_move_start=t1,
    t_move_end=t2,
)

In [ ]:
plot_distance_moved_vs_class_mean(
    stats["distances_to_manifolds"],
    stats["distance_moved"],
    slope=stats["slope"],
    intercept=stats["intercept"],
    pearson_r=stats["pearson_r"],
    dpi=300,
    show=True,
    box_w=0.6,
    box_h=0.6,
    max_y=1,
    # max_y=8,
)

### Generate ground truth data from this system

In [ ]:
# Shared student–teacher trial design (both hand-constructed teachers).
ST_TRIAL_DEFAULTS = {
    "n_trials": 6000,
    "n_conditions": 6,
    "n_sessions": 8,
    "train_perc": 0.75,
    "trial_duration_s": 4.0,
    "stim_onset_s": (0.5, 0.7),
    "stim_duration_s": 0.25,
    "observation_on": "currents",
}

# Observation gains tuned to match recorded marginal spike statistics.
ST_OBS_TRANSIENT = {"w_obs": 1.0, "bias_scale": 1.0, "bias_mean": -0.25}

dataset_kwargs = {**ST_TRIAL_DEFAULTS, **ST_OBS_TRANSIENT}

dataset_seed = int(np.random.randint(1e6)) if save else seed
st_data = generate_student_teacher_dataset(
    rnn,
    seed=dataset_seed,
    dataset_name="transient",
    save_path="../data/synthetic" if save else None,
    **dataset_kwargs,
)

y = st_data["y"]
trial_labels = st_data["labels"]
currents = st_data["currents"]
R_sub = st_data["rates"]
y_rates = st_data["y_rates"]
task_params = st_data["task_params"]
n_trials = y.shape[0]

In [ ]:
print(dataset_seed)

In [ ]:
# Spike histogram stats
# data stats
# mean population rate (Hz): 15.311
# mean ISI per unit (s): 0.171
# mean |pairwise corr|: 0.036

In [ ]:
stats_synth = spike_histogram_stats_from_array(y, task_params["bin_size"])
plot_spike_histogram_stats(
    stats_synth,
)

In [ ]:
plot_student_teacher_overview(
    currents,
    R_sub,
    y_rates,
    y,
    trial=0,
    unit_indices=(0, 100),
    box_w=1,
    box_h=0.6,
    panel_gap_x=0.3,
    panel_gap_y=0.5,
)

In [ ]:
plot_unit_activity_by_stimulus(
    st_data=st_data, stimulus=0, panel_gap_y=0.5, box_w=1, box_h=0.6
)